# Prime Class Characters

This notebook investigates how prime numbers distribute across Collatz
classification structures -- dropping sets, stopping classes, and orbital
oddity. Primes are the atoms of the integers, and understanding their
Collatz behaviour may reveal structural patterns that constrain all
composite orbits.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collatz.core import total_stopping_time
from collatz.dropping import dropping_set, orbital_oddity
from collatz.stopping import stopping_class, stopping_point
from collatz.geometry import complex_multiplier
from collatz.factorization import is_prime, shared_class_primes

%matplotlib inline
plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 11})
print("Imports ready.")

## Prime Distribution Across Dropping Sets

In [ ]:
# Classify all primes up to 10000 by dropping set
PRIME_LIMIT = 10000
primes = [p for p in range(2, PRIME_LIMIT + 1) if is_prime(p)]
print(f"Found {len(primes)} primes up to {PRIME_LIMIT}")

prime_ds = [dropping_set(p) for p in primes]

# Histogram
unique_ds = sorted(set(prime_ds))
ds_counts = {d: prime_ds.count(d) for d in unique_ds}

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(ds_counts.keys(), ds_counts.values(), color="mediumseagreen", edgecolor="white")
ax.set_xlabel("Dropping Set (k)")
ax.set_ylabel("Number of Primes")
ax.set_title(f"Prime Distribution Across Dropping Sets (primes up to {PRIME_LIMIT})")
plt.tight_layout()
plt.show()

print("\nTop 10 dropping sets by prime count:")
for d, c in sorted(ds_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  Set {d:>3}: {c:>4} primes")

## Primes by Stopping Time

In [ ]:
# Histogram of total stopping times for primes up to 10000
prime_tst = [total_stopping_time(p) for p in primes]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(prime_tst, bins=50, color="orchid", edgecolor="white")
axes[0].set_xlabel("Total Stopping Time")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Total Stopping Time Distribution for Primes up to {PRIME_LIMIT}")

axes[1].scatter(primes, prime_tst, s=1, alpha=0.4, color="orchid")
axes[1].set_xlabel("Prime p")
axes[1].set_ylabel("Total Stopping Time")
axes[1].set_title(f"Total Stopping Time vs Prime (up to {PRIME_LIMIT})")

plt.tight_layout()
plt.show()

print(f"Mean stopping time for primes: {np.mean(prime_tst):.1f}")
print(f"Max stopping time: {max(prime_tst)} at p = {primes[prime_tst.index(max(prime_tst))]}")

## Prime-Rich and Prime-Poor Classes

In [ ]:
# For each stopping class, count the number of primes. Bar chart.
prime_sc = [stopping_class(p) for p in primes]
sc_counts = {}
for sc in prime_sc:
    sc_counts[sc] = sc_counts.get(sc, 0) + 1

sorted_classes = sorted(sc_counts.keys())
sorted_counts = [sc_counts[c] for c in sorted_classes]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(sorted_classes, sorted_counts, color="salmon", edgecolor="white")
ax.set_xlabel("Stopping Class (k)")
ax.set_ylabel("Number of Primes")
ax.set_title(f"Prime Count by Stopping Class (primes up to {PRIME_LIMIT})")

# Highlight top 3
top3 = sorted(sc_counts.items(), key=lambda x: -x[1])[:3]
for cls, cnt in top3:
    idx = sorted_classes.index(cls)
    bars[idx].set_color("crimson")

plt.tight_layout()
plt.show()

print("Top 5 prime-rich stopping classes:")
for cls, cnt in sorted(sc_counts.items(), key=lambda x: -x[1])[:5]:
    print(f"  Class {cls:>3}: {cnt:>4} primes")
print("\nTop 5 prime-poor stopping classes (excluding empty):")
for cls, cnt in sorted(sc_counts.items(), key=lambda x: x[1])[:5]:
    print(f"  Class {cls:>3}: {cnt:>4} primes")

## Prime Density by Class

In [ ]:
# Compute prime density (primes / total members) for each stopping class up to 10000
# First, get stopping class for ALL numbers 2..10000
all_sc = {}
for n in range(2, PRIME_LIMIT + 1):
    sc = stopping_class(n)
    if sc not in all_sc:
        all_sc[sc] = {"total": 0, "primes": 0}
    all_sc[sc]["total"] += 1
    if is_prime(n):
        all_sc[sc]["primes"] += 1

# Compute density
density_data = []
for sc in sorted(all_sc.keys()):
    total = all_sc[sc]["total"]
    prime_count = all_sc[sc]["primes"]
    density = prime_count / total if total > 0 else 0
    density_data.append({"class": sc, "total": total, "primes": prime_count, "density": density})

df_density = pd.DataFrame(density_data)
# Filter to classes with at least 5 members for meaningful density
df_filtered = df_density[df_density["total"] >= 5].copy()

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(df_filtered["class"], df_filtered["density"], color="teal", edgecolor="white")
ax.axhline(len(primes) / (PRIME_LIMIT - 1), color="red", ls="--", lw=1.5,
           label=f"Overall prime density = {len(primes)/(PRIME_LIMIT-1):.4f}")
ax.set_xlabel("Stopping Class (k)")
ax.set_ylabel("Prime Density (primes / total in class)")
ax.set_title(f"Prime Density by Stopping Class (n up to {PRIME_LIMIT}, classes with >= 5 members)")
ax.legend()
plt.tight_layout()
plt.show()

print(df_filtered.to_string(index=False))

## Complex Multipliers of Primes

In [ ]:
# Plot z_0 for all primes up to 5000 on the complex plane, colored by stopping class
COMPLEX_LIMIT = 5000
primes_c = [p for p in primes if p <= COMPLEX_LIMIT]
z_vals = [complex_multiplier(p) for p in primes_c]
sc_vals = [stopping_class(p) for p in primes_c]

re_vals = [z.real for z in z_vals]
im_vals = [z.imag for z in z_vals]

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(re_vals, im_vals, c=sc_vals, cmap="tab20", s=10, alpha=0.7)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Stopping Class")

ax.set_xlabel("Re(z_0)")
ax.set_ylabel("Im(z_0)")
ax.set_title(f"Complex Multiplier z_0 for Primes up to {COMPLEX_LIMIT}")
ax.axvline(0.5, color="grey", ls=":", lw=0.8, label="Re = 1/2")
ax.legend()
plt.tight_layout()
plt.show()

print(f"All real parts = 0.5: {all(abs(z.real - 0.5) < 1e-12 for z in z_vals)}")
print(f"Imaginary part range: [{min(im_vals):.6f}, {max(im_vals):.6f}]")

## Observations

Initial patterns observed from this analysis:

1. **Dropping set 1 has no odd primes** -- by definition, all even numbers fall
   into dropping set 1, so only the prime 2 appears there. The remaining primes
   spread across the odd dropping sets.

2. **Dropping set 3 is prime-rich** -- a large proportion of primes land in
   dropping set 3 (all primes congruent to 1 mod 4 greater than 1). This is the
   smallest odd dropping set and captures many small primes.

3. **Complex multipliers all lie on Re = 1/2** -- this is a structural property
   of the mapping (not specific to primes), but the *distribution* of imaginary
   parts differs for primes vs composites. The imaginary parts appear to be
   denser near 0 and sparser near 1/2.

4. **Prime density varies significantly by stopping class** -- some classes are
   notably enriched or depleted in primes relative to the baseline density.
   Understanding why could connect Collatz dynamics to arithmetic structure.

5. **Stopping time distribution for primes** mirrors the overall distribution
   but with subtle differences -- primes may systematically avoid certain
   stopping times or favour others. Further investigation is needed.